<a target="_parent" href="https://colab.research.google.com/github/gretelai/gretel-blueprints/blob/main/docs/notebooks/data-designer/data-designer-101/3-seeding-with-a-dataset.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# 🎨 Data Designer 101: Seeding synthetic data generation with an external dataset

In this notebook, we will demonstrate how to seed synthetic data generation in `DataDesigner` with an external dataset.


If this is your first time using `DataDesigner`, we recommend starting with the [first notebook](https://github.com/gretelai/gretel-blueprints/blob/main/docs/notebooks/data-designer/data-designer-101/1-the-basics.ipynb) in this 101 series.


<br>

### 💾 Install `gretel-client` and its dependencies

In [1]:
%%capture
%pip install -U gretel_client datasets

In [2]:
from gretel_client.navigator_client import Gretel

# The Gretel object is the SDK's main entry point for interacting with Gretel's API.
gretel = Gretel(api_key="prompt")

Gretel API Key: ··········
Logged in as abhiraams2004@gmail.com ✅


INFO:gretel_client.navigator_client:Using project: default-sdk-project-8345632e7b003a1
INFO:gretel_client.navigator_client:Project link: https://console.gretel.ai/proj_31FnaQ6aYYzRjHLOPohN6j6SajD


## 🏥 Download a seed dataset

- For this notebook, we'll change gears and create a synthetic dataset of patient notes.

- To steer the generation process, we will use Gretel's open-source [symptom-to-diagnosis dataset](https://huggingface.co/datasets/gretelai/symptom_to_diagnosis).

In [3]:
from datasets import load_dataset

df_seed = load_dataset("gretelai/symptom_to_diagnosis")["train"].to_pandas()
df_seed = df_seed.rename(columns={"output_text": "diagnosis", "input_text": "patient_summary"})

print(f"Number of records: {len(df_seed)}")

df_seed.head()

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.jsonl: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/853 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/212 [00:00<?, ? examples/s]

Number of records: 853


,diagnosis,patient_summary
0,cervical spondylosis,I've been having a lot of pain in my neck and ...
1,impetigo,I have a rash on my face that is getting worse...
2,urinary tract infection,I have been urinating blood. I sometimes feel ...
3,arthritis,I have been having trouble with my muscles and...
4,dengue,I have been feeling really sick. My body hurts...


## 👩‍⚕️ Designing our synthetic patient notes dataset

- We set the seed dataset using the `with_seed_dataset` method.

- We use the `shuffle` sampling strategy, which shuffles the seed dataset before sampling.

- We set `with_replacement=False`, so our max num_records is 853,


In [4]:
aidd = gretel.data_designer.new(model_suite="apache-2.0")

aidd.with_seed_dataset(
    df_seed,
    sampling_strategy="shuffle",
    with_replacement=False
)

# Empty dictionaries mean use default settings for the person samplers.
aidd.with_person_samplers({"patient_sampler": {}, "doctor_sampler": {}})

[23:46:35] [INFO] 🌱 Using seed dataset with file ID: file_67c6230159294a1a8431ab51497267d6


DataDesigner(
    model_suite: apache-2.0
    person_samplers: ['patient_sampler', 'doctor_sampler']
    seed_dataset: file_67c6230159294a1a8431ab51497267d6
    seed_columns: ['diagnosis', 'patient_summary']
)

In [5]:
# Here we demonstrate how you can add a column by calling `add_column` with the
# column name, column type, and any parameters for that column type. This is in
# contrast to using the column and parameter type objects, via `C` and `P`, as we
# did in the previous notebook. Generally, we recommend using the concrete column
# and parameter type objects, but this is a convenient shorthand when you are
# familiar with the required arguments for each type.

aidd.add_column(
    name="patient_id",
    type="uuid",
    params={"prefix": "PT-", "short_form": True, "uppercase": True},
)

aidd.add_column(
    name="first_name",
    type="expression",
    expr="{{ patient_sampler.first_name}} ",
)

aidd.add_column(
    name="last_name",
    type="expression",
    expr="{{ patient_sampler.last_name }}",
)


aidd.add_column(
    name="dob",
    type="expression",
    expr="{{ patient_sampler.birth_date }}"
)


aidd.add_column(
    name="patient_email",
    type="expression",
    expr="{{ patient_sampler.email_address }}",
)


aidd.add_column(
    name="symptom_onset_date",
    type="datetime",
    params={"start": "2024-01-01", "end": "2024-12-31"},
)

aidd.add_column(
    name="date_of_visit",
    type="timedelta",
    params={
        "dt_min": 1,
        "dt_max": 30,
        "reference_column_name": "symptom_onset_date"
    },
)

aidd.add_column(
    name="physician",
    type="expression",
    expr="Dr. {{ doctor_sampler.last_name }}",
)

# Note we have access to the seed data fields.
aidd.add_column(
    name="physician_notes",
    prompt="""\
You are a primary-care physician who just had an appointment with {{ first_name }} {{ last_name }},
who has been struggling with symptoms from {{ diagnosis }} since {{ symptom_onset_date }}.
The date of today's visit is {{ date_of_visit }}.

{{ patient_summary }}

Write careful notes about your visit with {{ first_name }},
as Dr. {{ doctor_sampler.first_name }} {{ doctor_sampler.last_name }}.

Format the notes as a busy doctor might.
"""
 )


aidd.with_evaluation_report().validate()

[23:47:05] [INFO] Validation passed ✅


DataDesigner(
    model_suite: apache-2.0
    person_samplers: ['patient_sampler', 'doctor_sampler']
    seed_dataset: file_67c6230159294a1a8431ab51497267d6
    seed_columns: ['diagnosis', 'patient_summary']
    sampler_columns: ['patient_id', 'symptom_onset_date', 'date_of_visit']
    llm_text_columns: ['physician_notes']
    expression_columns: [
        "first_name",
        "last_name",
        "dob",
        "patient_email",
        "physician"
    ]
)

## 👀 Preview the dataset

- Iteration is key to generating high-quality synthetic data.

- Use the `preview` method to generate 10 records for inspection.

In [6]:
preview = aidd.preview()


[23:47:19] [INFO] 🚀 Generating preview
[23:47:20] [INFO] 🌱 Step 1: Seeding workflow with dataset
[23:47:21] [INFO] 🎲 Step 2: Using samplers to generate 5 columns
[23:47:22] [INFO] 🔗 Step 3: Concatenating seed and sampler datasets
[23:47:22] [INFO] 💬 Step 4: Rendering expression column `last_name`
[23:47:22] [INFO] 💬 Step 5: Rendering expression column `first_name`
[23:47:22] [INFO] 💬 Step 6: Rendering expression column `dob`
[23:47:23] [INFO] 💬 Step 7: Rendering expression column `patient_email`
[23:47:23] [INFO] 💬 Step 8: Rendering expression column `physician`
[23:47:23] [INFO] 🦜 Step 9: Generating text column `physician_notes`
[23:47:37] [INFO] 🙈 Step 10: Dropping 2 latent person columns
[23:47:37] [INFO] 🧐 Step 11: Evaluating dataset
[23:47:37] [INFO] 🎉 Your dataset preview is ready!


In [7]:
# The preview dataset is available as a pandas DataFrame.
preview.dataset.df.head()

,diagnosis,patient_summary,patient_id,symptom_onset_date,date_of_visit,last_name,first_name,dob,patient_email,physician,physician_notes
0,allergy,"I feel tired all the time, and my throat feels...",PT-015E707C,2024-06-22,2024-07-05,Cobo,Aida,1991-08-03,aida.cobo1991@icloud.com,Dr. Dupree,**Patient Notes**\n\n**Patient Name:** Aida Co...
1,typhoid,"I've been having diarrhea and constipation, wh...",PT-9EE747E7,2024-08-12,2024-08-13,Jones,Matthew,1940-12-27,matthewjones1940@gmail.com,Dr. Deleon,**Patient Visit Notes**\n\n**Patient:** Matthe...
2,gastroesophageal reflux disease,"I have a burning sensation in my throat, espec...",PT-A47D08BE,2024-04-06,2024-04-11,Moore,Allen,2007-07-05,amoore07@yahoo.com,Dr. Kidd,**Patient Visit Notes**\n\n**Patient:** Allen ...
3,jaundice,"I've been feeling scratchy, sick, and worn out...",PT-6444690E,2024-04-12,2024-04-17,Calvino,Francisco,1988-01-14,francisco_calvino@gmail.com,Dr. Mcteague,**Patient Visit Notes**\n\n**Patient Name:** F...
4,pneumonia,I've been feeling really unwell recently. I've...,PT-FBA7889E,2024-06-06,2024-06-07,Robinson,Beauregard,1976-06-18,beauregardr56@icloud.com,Dr. Gomez,**Patient Visit Notes**\n\n**Patient Name:** B...


In [8]:
# Run this cell multiple times to cycle through the 10 preview records.
preview.display_sample_record()

                                                   Seed Columns                                                    
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Name            ┃ Value                                                                                         ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ diagnosis       │ allergy                                                                                       │
├─────────────────┼───────────────────────────────────────────────────────────────────────────────────────────────┤
│ patient_summary │ I feel tired all the time, and my throat feels tingly. I also have dry, flaky skin. Sometimes │
│                 │ my eyes get puffy, and my face swells up too.                                                 │
└─────────────────┴───────────────────────────────────────────────────────────────────────────────────────────────┘
                                                                                                                   
                                                                                                                   
                                                 Generated Columns                                                 
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Name                       ┃ Value                                                                              ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ patient_id                 │ PT-015E707C                                                                        │
├────────────────────────────┼────────────────────────────────────────────────────────────────────────────────────┤
│ symptom_onset_date         │ 2024-06-22                                                                         │
├────────────────────────────┼────────────────────────────────────────────────────────────────────────────────────┤
│ date_of_visit              │ 2024-07-05                                                                         │
├────────────────────────────┼────────────────────────────────────────────────────────────────────────────────────┤
│ first_name                 │ Aida                                                                               │
├────────────────────────────┼────────────────────────────────────────────────────────────────────────────────────┤
│ last_name                  │ Cobo                                                                               │
├────────────────────────────┼────────────────────────────────────────────────────────────────────────────────────┤
│ dob                        │ 1991-08-03                                                                         │
├────────────────────────────┼────────────────────────────────────────────────────────────────────────────────────┤
│ patient_email              │ aida.cobo1991@icloud.com                                                           │
├────────────────────────────┼────────────────────────────────────────────────────────────────────────────────────┤
│ physician                  │ Dr. Dupree                                                                         │
├────────────────────────────┼────────────────────────────────────────────────────────────────────────────────────┤
│ physician_notes            │ **Patient Notes**                                                                  │
│                            │                                                                                    │
│                            │ **Patient Name:** Aida Cobo                                                        │
│                            │                          

## 🆙 Scale up!

- Once you are happy with the preview, scale up to a larger dataset by submitting a batch workflow.

- You can view the evaluation report by following the workflow link in the output of `create` below.

- Click the link to follow along with the generation process.

In [9]:
workflow_run = aidd.create(num_records=100, name="aidd-101-notebook-3-patient-notes")

[23:48:08] [INFO] 🚀 Submitting batch workflow


INFO:gretel_client.workflows.builder:▶️ Creating Workflow: w_31FnrncKJJ3gz8sGybTfjgI7WxK
INFO:gretel_client.workflows.builder:▶️ Created Workflow Run: wr_31FnrruCtH1hlCt0dtql2uZil6K
INFO:gretel_client.workflows.builder:🔗 Workflow Run console link: https://console.gretel.ai/workflows/w_31FnrncKJJ3gz8sGybTfjgI7WxK/runs/wr_31FnrruCtH1hlCt0dtql2uZil6K
